# Reviewer 1 Major Concern 3

"Missing-data bias, particularly in the Y2H network, is not addressed. The authors should examine whether proteins absent from each network differ systematically in abundance, localization, disorder, stability, essentiality, or other relevant characteristics. For example, long ORFs are harder to clone and are underrepresented in the interactome datasets."

In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from tqdm import tqdm, trange

In [2]:
DATA_PATH1 = "../../0-download-inputs/data-files/"
DATA_PATH2 = "../../18-finalize/processed-data/"
DATA_PATH3 = "../../16-Y2H-data/processed-data/"
MS_COLOR   = "#DDAA33"
Y2H_COLOR  = "#BB5566"

## Create out-of-network input file for annotation

In [3]:
# Read network node lists.
AEMS_nodes = pd.read_pickle(
    f"{DATA_PATH2}20260210-s288c-ANnnotated-Yeast-Interactome.pkl"
)
Y2H_nodes = pd.read_csv(f"{DATA_PATH3}Y2H-s288c-step16.csv")
Y2H_nodes.columns = Y2H_nodes.columns.str.strip()


def classify_network_nodes(values):
    """Separate individual identifiers from candidates in grouped nodes."""
    values = values.astype("string").str.strip().dropna()
    values = values.loc[values.ne("")]

    individual_ids = set()
    grouped_ids = set()
    grouped_rows = 0

    for value in values:
        identifiers = {
            identifier.strip()
            for identifier in value.split(";")
            if identifier.strip()
        }

        if ";" in value:
            grouped_rows += 1
            grouped_ids.update(identifiers)
        else:
            individual_ids.update(identifiers)

    # Individual identification takes precedence over grouped membership.
    ambiguous_ids = grouped_ids - individual_ids

    return individual_ids, ambiguous_ids, grouped_rows


in_AEMS_ids, ambiguous_AEMS_ids, AEMS_grouped_rows = classify_network_nodes(
    AEMS_nodes["node"]
)
in_Y2H_ids, ambiguous_Y2H_ids, Y2H_grouped_rows = classify_network_nodes(
    Y2H_nodes["node"]
)

# Extract ordered locus IDs from the first word of each FASTA header.
fasta_path = Path(DATA_PATH1) / "orf_trans.fasta"
reference_ids = set()
record_count = 0

with fasta_path.open(encoding="utf-8-sig") as handle:
    for line_number, line in enumerate(handle, start=1):
        if line.startswith(">"):
            header = line[1:].strip()
            if not header:
                raise ValueError(
                    f"Empty FASTA header at line {line_number} in {fasta_path}"
                )
            reference_ids.add(header.split()[0])
            record_count += 1

if not reference_ids:
    raise ValueError(f"No FASTA identifiers found in {fasta_path}")

# Confidently absent: neither individually identified nor a group candidate.
missing_AEMS = reference_ids - (in_AEMS_ids | ambiguous_AEMS_ids)
missing_Y2H = reference_ids - (in_Y2H_ids | ambiguous_Y2H_ids)

# Compute these proteins once, then use network-specific sets for comparisons.
missing_consensus = missing_AEMS | missing_Y2H

missing_AEMS_nodes = pd.DataFrame({"node": sorted(missing_AEMS)})
missing_Y2H_nodes = pd.DataFrame({"node": sorted(missing_Y2H)})
missing_nodes = pd.DataFrame({"node": sorted(missing_consensus)})

output_dir = Path("processed-data")
output_dir.mkdir(parents=True, exist_ok=True)

for frame, filename in (
    (missing_AEMS_nodes, "confidently_absent_AEMS_nodes.csv"),
    (missing_Y2H_nodes, "confidently_absent_Y2H_nodes.csv"),
    (missing_nodes, "missing_AEMS_or_Y2H_nodes.csv"),
):
    output_path = output_dir / filename
    frame.to_csv(output_path, index=False)
    print(f"{output_path}: {len(frame):,} loci")

print(f"\nFASTA records: {record_count:,}")
print(f"Unique reference loci: {len(reference_ids):,}")

for name, present, ambiguous, absent, grouped_rows in (
    ("AE-MS", in_AEMS_ids, ambiguous_AEMS_ids, missing_AEMS, AEMS_grouped_rows),
    ("Y2H", in_Y2H_ids, ambiguous_Y2H_ids, missing_Y2H, Y2H_grouped_rows),
):
    print(f"\n{name}")
    print(f"  Semicolon-delimited node rows: {grouped_rows:,}")
    print(f"  Unambiguously present reference loci: {len(present & reference_ids):,}")
    print(f"  Ambiguously represented reference loci: {len(ambiguous & reference_ids):,}")
    print(f"  Confidently absent reference loci: {len(absent):,}")
    print(
        "  Network identifiers absent from reference FASTA: "
        f"{len((present | ambiguous) - reference_ids):,}"
    )

processed-data/confidently_absent_AEMS_nodes.csv: 2,111 loci
processed-data/confidently_absent_Y2H_nodes.csv: 4,079 loci
processed-data/missing_AEMS_or_Y2H_nodes.csv: 4,568 loci

FASTA records: 6,039
Unique reference loci: 6,039

AE-MS
  Semicolon-delimited node rows: 41
  Unambiguously present reference loci: 3,876
  Ambiguously represented reference loci: 52
  Confidently absent reference loci: 2,111
  Network identifiers absent from reference FASTA: 11

Y2H
  Semicolon-delimited node rows: 0
  Unambiguously present reference loci: 1,960
  Ambiguously represented reference loci: 0
  Confidently absent reference loci: 4,079
  Network identifiers absent from reference FASTA: 58


## Functions for later use

In [ ]:
def bootstrap_ci_median(values, B=1_000_000, ci=95, random_state=27):
    """
    Nonparametric bootstrap CI for the median.

    values: 1D array-like
    B: number of bootstrap resamples
    ci: confidence level (e.g., 95)
    """
    vals = np.asarray(values)
    vals = vals[np.isfinite(vals)]
    n = len(vals)
    if n == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(random_state)
    medians = np.empty(B, dtype=float)

    # Loop-based bootstrap to avoid huge memory allocations
    for b in trange(B, desc="Bootstrapping medians"):
        sample = rng.choice(vals, size=n, replace=True)
        medians[b] = np.median(sample)

    median_hat = np.median(vals)
    alpha = 100 - ci
    lower = np.percentile(medians, alpha / 2)
    upper = np.percentile(medians, 100 - alpha / 2)
    return median_hat, lower, upper

## Abundance, AE-MS

In [ ]:
# Read network nodes and standardize identifiers.
AEMS_nodes = pd.read_pickle(
    f"{DATA_PATH2}20260210-s288c-ANnnotated-Yeast-Interactome.pkl"
)
node_ids = AEMS_nodes["node"].astype("string").str.strip()
in_AEMS_ids = set(node_ids.dropna().loc[lambda x: x.ne("")])

# Read raw abundance data.
abundances = pd.read_csv(
    f"{DATA_PATH1}1-s2.0-S240547121730546X-mmc5.csv"
)
abundances.columns = abundances.columns.str.strip()

abundance_ids = abundances["Systematic Name"].astype("string").str.strip()
abundance_values = pd.to_numeric(
    abundances["Median molecules per cell"], errors="coerce"
)

# Retain rows with a usable identifier and finite abundance measurement.
valid_mask = (
    abundance_ids.notna()
    & abundance_ids.ne("")
    & abundance_values.notna()
    & np.isfinite(abundance_values)
)

abundance_data = pd.DataFrame({
    "node": abundance_ids.loc[valid_mask],
    "Median molecules per cell": abundance_values.loc[valid_mask],
}).reset_index(drop=True)

# Require one measurement per protein rather than silently aggregating.
duplicate_ids = abundance_data.loc[
    abundance_data["node"].duplicated(keep=False), "node"
].unique()

if len(duplicate_ids):
    raise ValueError(
        f"Found {len(duplicate_ids):,} proteins with multiple abundance rows. "
        f"Resolve these before plotting. Examples: {duplicate_ids[:10].tolist()}"
    )

# Split abundance measurements by network membership.
inside_mask = abundance_data["node"].isin(in_AEMS_ids)
outside_mask = ~inside_mask

abundances_in_AEMS = abundance_data.loc[inside_mask].reset_index(drop=True)
abundances_out_AEMS = abundance_data.loc[outside_mask].reset_index(drop=True)

inside = abundances_in_AEMS["Median molecules per cell"]
outside = abundances_out_AEMS["Median molecules per cell"]

# Proteins outside the network with abundance measurements.
out_AEMS = abundances_out_AEMS["node"].tolist()

if inside.empty or outside.empty:
    raise ValueError("Both groups must contain abundance measurements.")

if (inside <= 0).any() or (outside <= 0).any():
    raise ValueError(
        "Nonpositive abundance values found; check before log plotting."
    )

print(f"In-network proteins with abundance data: {len(inside):,}")
print(f"Out-of-network proteins with abundance data: {len(outside):,}")
print(f"Rows excluded for missing IDs or invalid abundance: {(~valid_mask).sum():,}")

# bootstrap 95% confidence intervals for medians of abundance values
m1, lo1, hi1 = bootstrap_ci_median(inside, B=1_000_000, ci=95, random_state=27)
m2, lo2, hi2 = bootstrap_ci_median(outside, B=1_000_000, ci=95, random_state=27)

# test for difference between medians
_u, _p = mannwhitneyu(inside, outside, alternative="two-sided")

# print stats results
print ("AE-MS, inside versus outside")
print(f"Inside AE-MS network : median={m1:.2f}, 95% CI=({lo1:.2f}, {hi1:.2f}), n={len(inside)})")
print(f"Outside AE-MS network: median={m2:.2f}, 95% CI=({lo2:.2f}, {hi2:.2f}), n={len(outside)})")
print(f"U = {_u:.2f}, p = {_p:.4g}")

# Plot abundance distributions.
fig, ax = plt.subplots(figsize=(4.5, 4.5))

boxes = ax.boxplot(
    [inside, outside],
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
)

for box, color in zip(boxes["boxes"], [MS_COLOR, "#BBBBBB"]):
    box.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xticklabels([
    f"In network\n(n = {len(inside):,})",
    f"Out of network\n(n = {len(outside):,})",
])
ax.set_yscale("log")
ax.set_ylabel("Protein abundance (median molecules per cell)")
ax.set_title("AE-MS")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

## Abundance, Y2H

In [ ]:
# Read Y2H network nodes and standardize identifiers.
Y2H_nodes = pd.read_csv(f"{DATA_PATH3}Y2H-s288c-step16.csv")
Y2H_nodes.columns = Y2H_nodes.columns.str.strip()

node_ids = Y2H_nodes["node"].astype("string").str.strip()
in_Y2H_ids = set(node_ids.dropna().loc[lambda x: x.ne("")])

# Read raw abundance data.
abundances = pd.read_csv(
    f"{DATA_PATH1}1-s2.0-S240547121730546X-mmc5.csv"
)
abundances.columns = abundances.columns.str.strip()

abundance_ids = abundances["Systematic Name"].astype("string").str.strip()
abundance_values = pd.to_numeric(
    abundances["Median molecules per cell"], errors="coerce"
)

# Retain rows with a usable identifier and finite abundance measurement.
valid_mask = (
    abundance_ids.notna()
    & abundance_ids.ne("")
    & abundance_values.notna()
    & np.isfinite(abundance_values)
)

abundance_data = pd.DataFrame({
    "node": abundance_ids.loc[valid_mask],
    "Median molecules per cell": abundance_values.loc[valid_mask],
}).reset_index(drop=True)

# Require one measurement per protein rather than silently aggregating.
duplicate_ids = abundance_data.loc[
    abundance_data["node"].duplicated(keep=False), "node"
].unique()

if len(duplicate_ids):
    raise ValueError(
        f"Found {len(duplicate_ids):,} proteins with multiple abundance rows. "
        f"Resolve these before plotting. Examples: {duplicate_ids[:10].tolist()}"
    )

# Split abundance measurements by Y2H network membership.
inside_mask = abundance_data["node"].isin(in_Y2H_ids)
outside_mask = ~inside_mask

abundances_in_Y2H = abundance_data.loc[inside_mask].reset_index(drop=True)
abundances_out_Y2H = abundance_data.loc[outside_mask].reset_index(drop=True)

inside = abundances_in_Y2H["Median molecules per cell"]
outside = abundances_out_Y2H["Median molecules per cell"]

# Proteins outside the Y2H network with abundance measurements.
out_Y2H = abundances_out_Y2H["node"].tolist()

if inside.empty or outside.empty:
    raise ValueError("Both groups must contain abundance measurements.")

if (inside <= 0).any() or (outside <= 0).any():
    raise ValueError(
        "Nonpositive abundance values found; check before log plotting."
    )

print(f"In-network proteins with abundance data: {len(inside):,}")
print(f"Out-of-network proteins with abundance data: {len(outside):,}")
print(f"Rows excluded for missing IDs or invalid abundance: {(~valid_mask).sum():,}")

# bootstrap 95% confidence intervals for medians of abundance values
m1, lo1, hi1 = bootstrap_ci_median(inside, B=1_000_000, ci=95, random_state=27)
m2, lo2, hi2 = bootstrap_ci_median(outside, B=1_000_000, ci=95, random_state=27)

# test for difference between medians
_u, _p = mannwhitneyu(inside, outside, alternative="two-sided")

# print stats results
print ("Y2H, inside versus outside")
print(f"Inside Y2H network : median={m1:.2f}, 95% CI=({lo1:.2f}, {hi1:.2f}), n={len(inside)})")
print(f"Outside Y2H network: median={m2:.2f}, 95% CI=({lo2:.2f}, {hi2:.2f}), n={len(outside)})")
print(f"U = {_u:.2f}, p = {_p:.4g}")

# Plot abundance distributions.
fig, ax = plt.subplots(figsize=(4.5, 4.5))

boxes = ax.boxplot(
    [inside, outside],
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
)

for box, color in zip(boxes["boxes"], [Y2H_COLOR, "#BBBBBB"]):
    box.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xticklabels([
    f"In network\n(n = {len(inside):,})",
    f"Out of network\n(n = {len(outside):,})",
])
ax.set_yscale("log")
ax.set_ylabel("Protein abundance (molecules per cell)")
ax.set_title("Y2H")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

## Half-life, AE-MS

In [ ]:
# Read AE-MS network nodes.
AEMS_nodes = pd.read_pickle(
    f"{DATA_PATH2}20260210-s288c-ANnnotated-Yeast-Interactome.pkl"
)
node_ids = AEMS_nodes["node"].astype("string").str.strip()
in_AEMS_ids = set(node_ids.dropna().loc[lambda x: x.ne("")])

# Read half-life data and convert hours to minutes.
halflife = pd.read_csv(
    f"{DATA_PATH1}1-s2.0-S2405471217303411-mmc2.csv"
)
halflife.columns = halflife.columns.str.strip()

halflife_ids = halflife["Protein IDs"].astype("string").str.strip()
halflife_minutes = pd.to_numeric(
    halflife["t_12_avg"], errors="coerce"
) * 60.0

# Retain usable identifiers and finite half-life measurements.
valid_mask = (
    halflife_ids.notna()
    & halflife_ids.ne("")
    & halflife_minutes.notna()
    & np.isfinite(halflife_minutes)
)

halflife_data = pd.DataFrame({
    "node": halflife_ids.loc[valid_mask],
    "t_12_avg": halflife.loc[valid_mask, "t_12_avg"],
    "t_12_avg_minutes": halflife_minutes.loc[valid_mask],
}).reset_index(drop=True)

# Require one measurement per protein.
duplicate_ids = halflife_data.loc[
    halflife_data["node"].duplicated(keep=False), "node"
].unique()

if len(duplicate_ids):
    raise ValueError(
        f"Found {len(duplicate_ids):,} proteins with multiple half-life rows. "
        f"Resolve these before plotting. Examples: {duplicate_ids[:10].tolist()}"
    )

# Split by AE-MS membership.
inside_mask = halflife_data["node"].isin(in_AEMS_ids)

halflife_in_AEMS = halflife_data.loc[inside_mask].reset_index(drop=True)
halflife_out_AEMS = halflife_data.loc[~inside_mask].reset_index(drop=True)

inside = halflife_in_AEMS["t_12_avg_minutes"]
outside = halflife_out_AEMS["t_12_avg_minutes"]
out_AEMS_halflife = halflife_out_AEMS["node"].tolist()

if inside.empty or outside.empty:
    raise ValueError("Both groups must contain half-life measurements.")

if (inside <= 0).any() or (outside <= 0).any():
    raise ValueError(
        "Nonpositive half-life values found; check before log plotting."
    )

print(f"In-network proteins with half-life data: {len(inside):,}")
print(f"Out-of-network proteins with half-life data: {len(outside):,}")
print(f"Rows excluded for missing IDs or invalid half-life: {(~valid_mask).sum():,}")

# bootstrap 95% confidence intervals for medians 
m1, lo1, hi1 = bootstrap_ci_median(inside, B=1_000_000, ci=95, random_state=27)
m2, lo2, hi2 = bootstrap_ci_median(outside, B=1_000_000, ci=95, random_state=27)

# test for difference between medians
_u, _p = mannwhitneyu(inside, outside, alternative="two-sided")

# print stats results
print ("AE-MS, inside versus outside")
print(f"Inside AE-MS network : median={m1:.2f}, 95% CI=({lo1:.2f}, {hi1:.2f}), n={len(inside)})")
print(f"Outside AE-MS network: median={m2:.2f}, 95% CI=({lo2:.2f}, {hi2:.2f}), n={len(outside)})")
print(f"U = {_u:.2f}, p = {_p:.4g}")

# Plot half-life distributions in minutes.
fig, ax = plt.subplots(figsize=(4.5, 4.5))

boxes = ax.boxplot(
    [inside, outside],
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
)

for box, color in zip(boxes["boxes"], [MS_COLOR, "#BBBBBB"]):
    box.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xticklabels([
    f"In network\n(n = {len(inside):,})",
    f"Out of network\n(n = {len(outside):,})",
])
ax.set_yscale("linear")
ax.set_ylabel("Protein half-life (minutes)")
ax.set_title("AE-MS")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

## Half-life, Y2H

In [ ]:
# Read Y2H network nodes.
Y2H_nodes = pd.read_csv(f"{DATA_PATH3}Y2H-s288c-step16.csv")
Y2H_nodes.columns = Y2H_nodes.columns.str.strip()

node_ids = Y2H_nodes["node"].astype("string").str.strip()
in_Y2H_ids = set(node_ids.dropna().loc[lambda x: x.ne("")])

# Read half-life data and convert hours to minutes.
halflife = pd.read_csv(
    f"{DATA_PATH1}1-s2.0-S2405471217303411-mmc2.csv"
)
halflife.columns = halflife.columns.str.strip()

halflife_ids = halflife["Protein IDs"].astype("string").str.strip()
halflife_minutes = pd.to_numeric(
    halflife["t_12_avg"], errors="coerce"
) * 60.0

# Retain usable identifiers and finite half-life measurements.
valid_mask = (
    halflife_ids.notna()
    & halflife_ids.ne("")
    & halflife_minutes.notna()
    & np.isfinite(halflife_minutes)
)

halflife_data = pd.DataFrame({
    "node": halflife_ids.loc[valid_mask],
    "t_12_avg": halflife.loc[valid_mask, "t_12_avg"],
    "t_12_avg_minutes": halflife_minutes.loc[valid_mask],
}).reset_index(drop=True)

# Require one measurement per protein.
duplicate_ids = halflife_data.loc[
    halflife_data["node"].duplicated(keep=False), "node"
].unique()

if len(duplicate_ids):
    raise ValueError(
        f"Found {len(duplicate_ids):,} proteins with multiple half-life rows. "
        f"Resolve these before plotting. Examples: {duplicate_ids[:10].tolist()}"
    )

# Split by Y2H membership.
inside_mask = halflife_data["node"].isin(in_Y2H_ids)

halflife_in_Y2H = halflife_data.loc[inside_mask].reset_index(drop=True)
halflife_out_Y2H = halflife_data.loc[~inside_mask].reset_index(drop=True)

inside = halflife_in_Y2H["t_12_avg_minutes"]
outside = halflife_out_Y2H["t_12_avg_minutes"]
out_Y2H_halflife = halflife_out_Y2H["node"].tolist()

if inside.empty or outside.empty:
    raise ValueError("Both groups must contain half-life measurements.")

if (inside <= 0).any() or (outside <= 0).any():
    raise ValueError(
        "Nonpositive half-life values found; check before log plotting."
    )

print(f"In-network proteins with half-life data: {len(inside):,}")
print(f"Out-of-network proteins with half-life data: {len(outside):,}")
print(f"Rows excluded for missing IDs or invalid half-life: {(~valid_mask).sum():,}")

# bootstrap 95% confidence intervals for medians 
m1, lo1, hi1 = bootstrap_ci_median(inside, B=1_000_000, ci=95, random_state=27)
m2, lo2, hi2 = bootstrap_ci_median(outside, B=1_000_000, ci=95, random_state=27)

# test for difference between medians
_u, _p = mannwhitneyu(inside, outside, alternative="two-sided")

# print stats results
print ("Y2H, inside versus outside")
print(f"Inside Y2H network : median={m1:.2f}, 95% CI=({lo1:.2f}, {hi1:.2f}), n={len(inside)})")
print(f"Outside Y2H network: median={m2:.2f}, 95% CI=({lo2:.2f}, {hi2:.2f}), n={len(outside)})")
print(f"U = {_u:.2f}, p = {_p:.4g}")

# Plot half-life distributions in minutes.
fig, ax = plt.subplots(figsize=(4.5, 4.5))

boxes = ax.boxplot(
    [inside, outside],
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
)

for box, color in zip(boxes["boxes"], [Y2H_COLOR, "#BBBBBB"]):
    box.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xticklabels([
    f"In network\n(n = {len(inside):,})",
    f"Out of network\n(n = {len(outside):,})",
])
ax.set_yscale("linear")
ax.set_ylabel("Protein half-life (minutes)")
ax.set_title("Y2H")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

## Melting point, AE-MS

* data preparation code cell

In [ ]:
# Read UniProt-to-ordered-locus mappings.
mapping = pd.read_csv(
    f"{DATA_PATH1}YEAST_559292_idmapping.dat",
    sep="\t",
    header=None,
    names=["UniProtKB-AC", "mapping_type", "node"],
    dtype=str,
    keep_default_na=False,
)

for column in mapping.columns:
    mapping[column] = mapping[column].str.strip()

mapping = (
    mapping.loc[
        mapping["mapping_type"].eq("Gene_OrderedLocusName")
        & mapping["UniProtKB-AC"].ne("")
        & mapping["node"].ne(""),
        ["UniProtKB-AC", "node"],
    ]
    .drop_duplicates()
)

# Read melting point measurements.
meltome = pd.read_csv(
    f"{DATA_PATH1}meltome-atlas-ma_0010.txt",
    sep="\t",
    dtype=str,
)
meltome.columns = meltome.columns.str.strip()

# For example: A5Z2X5_YPR010C-A -> A5Z2X5.
accessions = (
    meltome["Protein ID"]
    .astype("string")
    .str.strip()
    .str.split("_", n=1)
    .str[0]
    .str.strip()
)
temperatures = pd.to_numeric(
    meltome["Melting point [°C]"], errors="coerce"
)

valid_mask = (
    accessions.notna()
    & accessions.ne("")
    & temperatures.notna()
    & np.isfinite(temperatures)
)

meltome_data = pd.DataFrame({
    "Protein ID": meltome.loc[valid_mask, "Protein ID"],
    "UniProtKB-AC": accessions.loc[valid_mask],
    "Melting point [°C]": temperatures.loc[valid_mask],
}).reset_index(drop=True)

# Require one source measurement per accession before expanding mappings.
duplicate_accessions = meltome_data.loc[
    meltome_data["UniProtKB-AC"].duplicated(keep=False), "UniProtKB-AC"
].unique()

if len(duplicate_accessions):
    raise ValueError(
        f"Found {len(duplicate_accessions):,} accessions with multiple "
        "source melting point rows. Resolve these before proceeding. "
        f"Examples: {duplicate_accessions[:10].tolist()}"
    )

relevant_mapping = mapping.loc[
    mapping["UniProtKB-AC"].isin(meltome_data["UniProtKB-AC"])
]

mapping_counts = relevant_mapping.groupby("UniProtKB-AC")["node"].nunique()
multiple_loci = mapping_counts.loc[mapping_counts.gt(1)]

# Expand each accession to all corresponding ordered loci.
# Each mapped locus receives the accession's original melting temperature.
meltome_data = meltome_data.merge(
    relevant_mapping,
    on="UniProtKB-AC",
    how="left",
    validate="one_to_many",
)

unmapped_mask = meltome_data["node"].isna()
meltome_unmapped = meltome_data.loc[unmapped_mask].copy()
meltome_data = meltome_data.loc[~unmapped_mask].reset_index(drop=True)

# Ensure each ordered locus still contributes only one row.
duplicate_ids = meltome_data.loc[
    meltome_data["node"].duplicated(keep=False), "node"
].unique()

if len(duplicate_ids):
    raise ValueError(
        f"Found {len(duplicate_ids):,} loci with multiple melting point rows "
        f"after mapping. Examples: {duplicate_ids[:10].tolist()}"
    )

print(f"Rows excluded for missing IDs or invalid temperature: {(~valid_mask).sum():,}")
print(f"Measurement rows excluded without locus mappings: {len(meltome_unmapped):,}")
print(f"Accessions expanded to multiple loci: {len(multiple_loci):,}")
print(f"Additional locus rows created: {int((multiple_loci - 1).sum()):,}")
print(f"Unique loci with mapped melting points: {len(meltome_data):,}")

In [ ]:
# Read AE-MS network nodes.
AEMS_nodes = pd.read_pickle(
    f"{DATA_PATH2}20260210-s288c-ANnnotated-Yeast-Interactome.pkl"
)
node_ids = AEMS_nodes["node"].astype("string").str.strip()
in_AEMS_ids = set(node_ids.dropna().loc[lambda x: x.ne("")])

# Split melting points by ordered-locus membership.
inside_mask = meltome_data["node"].isin(in_AEMS_ids)

meltome_in_AEMS = meltome_data.loc[inside_mask].reset_index(drop=True)
meltome_out_AEMS = meltome_data.loc[~inside_mask].reset_index(drop=True)

inside = meltome_in_AEMS["Melting point [°C]"]
outside = meltome_out_AEMS["Melting point [°C]"]
out_AEMS_meltome = meltome_out_AEMS["node"].tolist()

if inside.empty or outside.empty:
    raise ValueError("Both groups must contain melting point measurements.")

print(f"In-network proteins with melting point data: {len(inside):,}")
print(f"Out-of-network proteins with melting point data: {len(outside):,}")

# bootstrap 95% confidence intervals for medians 
m1, lo1, hi1 = bootstrap_ci_median(inside, B=1_000_000, ci=95, random_state=27)
m2, lo2, hi2 = bootstrap_ci_median(outside, B=1_000_000, ci=95, random_state=27)

# test for difference between medians
_u, _p = mannwhitneyu(inside, outside, alternative="two-sided")

# print stats results
print ("AE-MS, inside versus outside")
print(f"Inside AE-MS network : median={m1:.2f}, 95% CI=({lo1:.2f}, {hi1:.2f}), n={len(inside)})")
print(f"Outside AE-MS network: median={m2:.2f}, 95% CI=({lo2:.2f}, {hi2:.2f}), n={len(outside)})")
print(f"U = {_u:.2f}, p = {_p:.4g}")

fig, ax = plt.subplots(figsize=(4.5, 4.5))

boxes = ax.boxplot(
    [inside, outside],
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
)

for box, color in zip(boxes["boxes"], [MS_COLOR, "#BBBBBB"]):
    box.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xticklabels([
    f"In network\n(n = {len(inside):,})",
    f"Out of network\n(n = {len(outside):,})",
])
ax.set_yscale("linear")
ax.set_ylabel("Melting point (°C)")
ax.set_title("AE-MS")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

## Melting point, Y2H

In [ ]:
# Read Y2H network nodes.
Y2H_nodes = pd.read_csv(f"{DATA_PATH3}Y2H-s288c-step16.csv")
Y2H_nodes.columns = Y2H_nodes.columns.str.strip()

node_ids = Y2H_nodes["node"].astype("string").str.strip()
in_Y2H_ids = set(node_ids.dropna().loc[lambda x: x.ne("")])

# Split melting points by ordered-locus membership.
inside_mask = meltome_data["node"].isin(in_Y2H_ids)

meltome_in_Y2H = meltome_data.loc[inside_mask].reset_index(drop=True)
meltome_out_Y2H = meltome_data.loc[~inside_mask].reset_index(drop=True)

inside = meltome_in_Y2H["Melting point [°C]"]
outside = meltome_out_Y2H["Melting point [°C]"]
out_Y2H_meltome = meltome_out_Y2H["node"].tolist()

if inside.empty or outside.empty:
    raise ValueError("Both groups must contain melting point measurements.")

print(f"In-network proteins with melting point data: {len(inside):,}")
print(f"Out-of-network proteins with melting point data: {len(outside):,}")

# bootstrap 95% confidence intervals for medians 
m1, lo1, hi1 = bootstrap_ci_median(inside, B=1_000_000, ci=95, random_state=27)
m2, lo2, hi2 = bootstrap_ci_median(outside, B=1_000_000, ci=95, random_state=27)

# test for difference between medians
_u, _p = mannwhitneyu(inside, outside, alternative="two-sided")

# print stats results
print ("Y2H, inside versus outside")
print(f"Inside Y2H network : median={m1:.2f}, 95% CI=({lo1:.2f}, {hi1:.2f}), n={len(inside)})")
print(f"Outside Y2H network: median={m2:.2f}, 95% CI=({lo2:.2f}, {hi2:.2f}), n={len(outside)})")
print(f"U = {_u:.2f}, p = {_p:.4g}")

fig, ax = plt.subplots(figsize=(4.5, 4.5))

boxes = ax.boxplot(
    [inside, outside],
    positions=[1, 2],
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.5},
)

for box, color in zip(boxes["boxes"], [Y2H_COLOR, "#BBBBBB"]):
    box.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xticklabels([
    f"In network\n(n = {len(inside):,})",
    f"Out of network\n(n = {len(outside):,})",
])
ax.set_yscale("linear")
ax.set_ylabel("Melting point (°C)")
ax.set_title("Y2H")
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()